# Generator smoke test

Tests the local LLM generator end-to-end: prompt rendering → Ollama call → fence stripping → `GeneratorResult`.

**Before running:** make sure Ollama is up (`ollama serve`) and the kernel is set to `.venv` (the UV workspace venv at `tried/.venv`).

No verification server or Azure judge needed here.

In [2]:
import os
os.environ["TRIED_ROLE"] = "orchestrator"

from collections import Counter
from pathlib import Path

from shared.dataset import load_corpus_train
from shared.logging import get_logger
from orchestrator.clients.generator_client import generate

_log = get_logger(__name__)

## 1 — Corpus overview

In [3]:
CORPUS_PATH = Path("../data/corpus_train.jsonl")

records = load_corpus_train(CORPUS_PATH)
print(f"{len(records)} training records loaded\n")

cats = Counter(r.op_category.value for r in records)
print(f"{'op_category':<35} count")
print("-" * 42)
for cat, count in sorted(cats.items(), key=lambda x: -x[1]):
    print(f"{cat:<35} {count}")

190 training records loaded

op_category                         count
------------------------------------------
matmul                              28
elementwise_chain                   23
normalization                       23
reduction                           20
convolution                         20
activation                          17
embedding                           16
fused_attention                     16
loss                                16
quantization                        11


## 2 — Pick a record

Change `RECORD_INDEX` to try different examples, or filter by `op_category` using the cell below.

In [4]:
RECORD_INDEX = 0

record = records[RECORD_INDEX]

print(f"example_id  : {record.example_id}")
print(f"origin      : {record.origin}")
print(f"op_category : {record.op_category.value}")
print(f"input_shapes: {record.input_shapes}")
print(f"input_dtypes: {[d.value for d in record.input_dtypes]}")
print(f"rng_seed    : {record.rng_seed}")
print()
print("PyTorch code:")
print("─" * 60)
print(record.pytorch_code)

example_id  : 276422e2-1ee2-56fd-96ca-d248a831ab29
origin      : curated/train/elementwise_clamp_square_fp32
op_category : elementwise_chain
input_shapes: [[1048576]]
input_dtypes: ['float32']
rng_seed    : 42

PyTorch code:
────────────────────────────────────────────────────────────
import torch

def op(x):
    y = torch.clamp(x, -3.0, 3.0)
    return y * y + 0.25 * y



In [ ]:
# Optional: filter records by op_category and pick from that subset
TARGET_CATEGORY = "reduction"  # change as needed

subset = [r for r in records if r.op_category.value == TARGET_CATEGORY]
print(f"{len(subset)} records with op_category='{TARGET_CATEGORY}'")
for i, r in enumerate(subset):
    print(f"  [{i}] {r.origin}  shapes={r.input_shapes}  dtypes={[d.value for d in r.input_dtypes]}")

## 3 — Generate (attempt 0, no prior advice)

In [5]:
print(f"Generating for: {record.example_id}")
print("(This can take 20–60s on first call while the model warms up)\n")

result = generate(
    pytorch_code=record.pytorch_code,
    input_shapes=record.input_shapes,
    input_dtypes=[d.value for d in record.input_dtypes],
)

print(f"prompt tokens      : {result.prompt_tokens}")
print(f"completion tokens  : {result.completion_tokens}")
print(f"latency            : {result.latency_ms}ms ({result.latency_ms / 1000:.1f}s)")
print()
print("Generated code:")
print("─" * 60)
print(result.triton_code)

Generating for: 276422e2-1ee2-56fd-96ca-d248a831ab29
(This can take 20–60s on first call while the model warms up)

prompt tokens      : 415
completion tokens  : 216
latency            : 8936ms (8.9s)

Generated code:
────────────────────────────────────────────────────────────
import triton
import triton.language as tl
import torch

@triton.jit
def op_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=-3.0)
    y = tl.clamp(x, -3.0, 3.0)
    result = y * y + 0.25 * y
    tl.store(y_ptr + offsets, result, mask=mask)

def op(x):
    n_elements = x.numel()
    y = torch.empty_like(x)
    BLOCK_SIZE = 1024
    grid = lambda meta: (triton.cdiv(n_elements, BLOCK_SIZE),)
    op_kernel[grid](x, y, n_elements, BLOCK_SIZE=BLOCK_SIZE)
    return y


## 4 — Generate with prior advice (simulates a retry)

Checks that the retry path renders the `prior_advice_section` into the user prompt correctly.
Edit `PRIOR_ADVICE` to whatever advice string you want to inject.

In [6]:
PRIOR_ADVICE = (
    "The previous attempt had incorrect indexing. "
    "Use tl.arange(0, BLOCK_SIZE) and add masking for the tail block "
    "so out-of-bounds lanes do not corrupt output."
)

result_retry = generate(
    pytorch_code=record.pytorch_code,
    input_shapes=record.input_shapes,
    input_dtypes=[d.value for d in record.input_dtypes],
    prior_advice=PRIOR_ADVICE,
)

print(f"prompt tokens      : {result_retry.prompt_tokens}")
print(f"completion tokens  : {result_retry.completion_tokens}")
print(f"latency            : {result_retry.latency_ms}ms ({result_retry.latency_ms / 1000:.1f}s)")
print()
print("Generated code (with prior advice):")
print("─" * 60)
print(result_retry.triton_code)

prompt tokens      : 457
completion tokens  : 215
latency            : 9045ms (9.0s)

Generated code (with prior advice):
────────────────────────────────────────────────────────────
import triton
import triton.language as tl
import torch

@triton.jit
def op_kernel(x_ptr, y_ptr, n_elements: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
    y = tl.clamp(x, -3.0, 3.0)
    result = y * y + 0.25 * y
    tl.store(y_ptr + offsets, result, mask=mask)

def op(x):
    n_elements = x.numel()
    y = torch.empty_like(x)
    grid = lambda meta: (triton.cdiv(n_elements, BLOCK_SIZE),)
    op_kernel[grid](x, y, n_elements=n_elements)
    return y

BLOCK_SIZE = 1024


## 5 — Diff attempt 0 vs retry

Quick visual check that the prior advice actually changed the output.

In [7]:
import difflib

diff = difflib.unified_diff(
    result.triton_code.splitlines(keepends=True),
    result_retry.triton_code.splitlines(keepends=True),
    fromfile="attempt_0",
    tofile="retry",
)
diff_text = "".join(diff)

if diff_text:
    print(diff_text)
else:
    print("No diff — model produced identical output (possible with temperature=0 and similar prompts)")

--- attempt_0
+++ retry
@@ -3,11 +3,12 @@
 import torch
 
 @triton.jit
-def op_kernel(x_ptr, y_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
+def op_kernel(x_ptr, y_ptr, n_elements: tl.constexpr):
     pid = tl.program_id(axis=0)
-    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
+    block_start = pid * BLOCK_SIZE
+    offsets = block_start + tl.arange(0, BLOCK_SIZE)
     mask = offsets < n_elements
-    x = tl.load(x_ptr + offsets, mask=mask, other=-3.0)
+    x = tl.load(x_ptr + offsets, mask=mask, other=0.0)
     y = tl.clamp(x, -3.0, 3.0)
     result = y * y + 0.25 * y
     tl.store(y_ptr + offsets, result, mask=mask)
@@ -15,7 +16,8 @@
 def op(x):
     n_elements = x.numel()
     y = torch.empty_like(x)
-    BLOCK_SIZE = 1024
     grid = lambda meta: (triton.cdiv(n_elements, BLOCK_SIZE),)
-    op_kernel[grid](x, y, n_elements, BLOCK_SIZE=BLOCK_SIZE)
-    return y+    op_kernel[grid](x, y, n_elements=n_elements)
+    return y
+
+BLOCK_SIZE = 1024
